# notebook_init — bootstrap for every pipeline notebook

Every Bronze / Silver / Gold notebook starts with:

```python
%run "../../libs/notebook_init"
```

The `%run` magic must be the **first character of the cell** — no comments or blank lines above it (see `.claude/commands/bundle-deploy.md`).

## What this notebook injects into the caller's namespace

- `CATALOG`, `BRONZE`, `SILVER`, `GOLD`, `AUDIT`, `RAW_FILES`
- Per-source raw paths: `RAW_ZILLOW`, `RAW_REALTOR`, `RAW_FHFA`, `RAW_FRED`
- Status literals: `STATUS_RUNNING`, `STATUS_SUCCEEDED`, `STATUS_FAILED`, `STATUS_NO_FILES`, `STATUS_SKIPPED`
- `PIPELINE_RUN_ID`
- Helpers: `Utils`, `pipeline_step_log_upsert`, `ingestion_log_insert`, `transform_detail_log_insert`
- Common imports: `uuid`, `time`, `sys`, `datetime`, `timezone`, `F` (= `pyspark.sql.functions`), `StructType`/`StructField`/types

## Catalog resolution (multi-target DAB)

`CATALOG` is resolved at runtime in this priority order:

1. **Job runs** — bundle injects `${var.catalog}` into the `catalog` widget.
2. **Manual runs from a deployed notebook** — path is inspected for the target name (`/marketpulse/<target>/files/...`); `_target_catalog_map` translates to the catalog name.
3. **Fallback** — `marketpulse` (prod).

`_target_catalog_map` MUST mirror the `catalog:` variable overrides in `databricks.yml`. If you rename a catalog, update both places. See `.claude/project_config.md`.

In [ ]:
import sys, uuid, time
from datetime import datetime, timezone
from pyspark.sql import Row, functions as F
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType, DoubleType, BooleanType,
    DateType, TimestampType,
)

# ---------------------------------------------------------------------------
# 1. Discover the notebook's own path; derive shared_lib_path for sys.path.
# Bundle deploys to: /Workspace/<user>/.bundle/marketpulse/<target>/files/libs/notebook_init
# ---------------------------------------------------------------------------
try:
    _nb_path = (
        dbutils.notebook.entry_point.getDbutils()
            .notebook().getContext().notebookPath().get()
    )
    if "/files/" in _nb_path:
        _default_lib = _nb_path.split("/files/")[0] + "/files/libs"
        if not _default_lib.startswith("/Workspace"):
            _default_lib = "/Workspace" + _default_lib
    else:
        _default_lib = ""
except Exception:
    _nb_path = ""
    _default_lib = ""

dbutils.widgets.text("shared_lib_path", _default_lib)
shared_lib_path = dbutils.widgets.get("shared_lib_path")
if shared_lib_path and shared_lib_path not in sys.path:
    sys.path.insert(0, shared_lib_path)

# ---------------------------------------------------------------------------
# 2. Catalog resolution.
# _target_catalog_map MUST mirror databricks.yml `targets:` variable overrides.
# Renaming a catalog requires updating BOTH places.
# ---------------------------------------------------------------------------
_target_catalog_map = {
    "dev":     "dev_marketpulse",
    "staging": "staging_marketpulse",
    "prod":    "marketpulse",
}

try:
    if "/marketpulse/" in _nb_path and "/files/" in _nb_path:
        _target = _nb_path.split("/marketpulse/")[1].split("/files/")[0]
        _targetcatalog = _target_catalog_map.get(_target, "marketpulse")
    else:
        _targetcatalog = "marketpulse"
except Exception:
    _targetcatalog = "marketpulse"

dbutils.widgets.text("catalog", _targetcatalog)
CATALOG     = dbutils.widgets.get("catalog")
BRONZE      = f"{CATALOG}.bronze"
SILVER      = f"{CATALOG}.silver"
GOLD        = f"{CATALOG}.gold"
AUDIT       = f"{CATALOG}.audit"
RAW_FILES   = f"/Volumes/{CATALOG}/raw/"
RAW_ZILLOW  = f"{RAW_FILES}zillow/"
RAW_REALTOR = f"{RAW_FILES}realtor/"
RAW_FHFA    = f"{RAW_FILES}fhfa/"
RAW_FRED    = f"{RAW_FILES}fred/"

# ---------------------------------------------------------------------------
# 3. Status literals.
# May move into pipeline_logging.py during migration Step 8 (audit-schema work).
# ---------------------------------------------------------------------------
STATUS_RUNNING   = "running"
STATUS_SUCCEEDED = "succeeded"
STATUS_FAILED    = "failed"
STATUS_NO_FILES  = "no_files"   # source folder empty; clean exit, not a failure
STATUS_SKIPPED   = "skipped"    # upstream failed but this task ran cleanup

# ---------------------------------------------------------------------------
# 4. Helper imports.
# Stubs exist now; bodies populated by migration Steps 2 and 8.
# ---------------------------------------------------------------------------
from pipeline_utils import Utils
from pipeline_logging import (
    pipeline_step_log_upsert,
    ingestion_log_insert,
    transform_detail_log_insert,
)

In [ ]:
# ---------------------------------------------------------------------------
# PIPELINE_RUN_ID — shared across all notebooks in a single pipeline run.
# Priority: explicit widget value > job init-task value > local fallback.
# ---------------------------------------------------------------------------
LOCAL_PIPELINE_ID = "11111"   # fallback when run manually outside a job

dbutils.widgets.text("pipeline_run_id", "")
_value = dbutils.widgets.get("pipeline_run_id")

if not _value:
    try:
        _value = str(dbutils.jobs.taskValues.get(
            taskKey    = "init_pipeline_log",
            key        = "pipeline_run_id",
            debugValue = LOCAL_PIPELINE_ID,
        ))
    except Exception:
        _value = str(LOCAL_PIPELINE_ID)

PIPELINE_RUN_ID = _value

print(f"notebook_init OK | CATALOG={CATALOG} | shared_lib_path={shared_lib_path or '<unset>'} | PIPELINE_RUN_ID={PIPELINE_RUN_ID}")